# ChromaDB vs CyborgDB Vec2Text Vulnerability Comparison

> **Last validated: 2026-06-04 | CyborgDB v0.17 (disk-backed)**

This notebook demonstrates the vec2text attack on both ChromaDB and CyborgDB side-by-side.

CyborgDB 0.17 ships as a single-node encrypted index that persists to local disk (RocksDB under the hood). The attack surface for an exfiltrated CyborgDB store is the on-disk SST files; this notebook reads those files raw, just like an attacker with filesystem access would.
 
### Attack Chain:
1. Sensitive texts → OpenAI embeddings → Store in BOTH ChromaDB and CyborgDB
2. Extract raw bytes from each database backend (ChromaDB SQLite, CyborgDB on-disk RocksDB)
3. Use vec2text to attempt reconstruction of original sensitive text
4. Compare results: ChromaDB (vulnerable) vs CyborgDB (protected)

**NOTE**: You will need an OpenAI API key. Set it as your `OPENAI_API_KEY` environment variable or you will be prompted in cell `#1`.

In [ ]:
# 0. Install dependencies
# Uninstall sentence-transformers first to avoid conflict with older transformers
%pip uninstall -y sentence-transformers --quiet

# Pinned to the versions validated on 2026-06-10 (Python 3.11, macOS arm64).
# - numpy<2 keeps the C-extension ABI happy.
# - transformers 4.57.6 / tokenizers 0.22.2 are vec2text 0.0.13's working pair.
# - rocksdict gives direct read access to CyborgDB's on-disk RocksDB store
#   for the "attacker reads the SSTs" demo in cell 6.
%pip install --quiet \
    "numpy==1.26.4" \
    "transformers==4.57.6" \
    "tokenizers==0.22.2" \
    "vec2text==0.0.13" \
    "openai==2.41.0" \
    "chromadb==1.5.9" \
    "getpass4==0.0.14.1" \
    "rocksdict==0.3.29" \
    "cyborgdb-core==0.17.0" \

# CPU-only torch — pinned to 2.12.0 (>=2.6 needed for CVE-2025-32434 fix in torch.load).
%pip install --quiet "torch==2.12.0" --index-url https://download.pytorch.org/whl/cpu


In [ ]:
# 1. Set up OpenAI embedding & vec2text corrector models

import os
import time

print("[1/5] Importing vec2text (pulls in torch/transformers — can take 30-60s on first run)...", flush=True)
_t = time.time()
import vec2text
print(f"      done in {time.time() - _t:.1f}s", flush=True)

print("[2/5] Importing OpenAI client...", flush=True)
from openai import OpenAI
import getpass
print("      done", flush=True)

# Environment variable setup
print("[3/5] Setting thread/MPS env vars...", flush=True)
def setup_env():
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1' 
    os.environ['MKL_NUM_THREADS'] = '1'
    os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
    os.environ['NUMEXPR_NUM_THREADS'] = '1'
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
setup_env()
print("      done", flush=True)

# ANSI color codes for live demo
class Colors:
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    END = '\033[0m'

def print_colored(text, color="", bold=False):
    """Print colored text for demo"""
    prefix = Colors.BOLD if bold else ""
    prefix += getattr(Colors, color.upper(), "")
    print(f"{prefix}{text}{Colors.END}")

# OpenAI setup
print("[4/5] Configuring OpenAI client...", flush=True)
embedding_model = "text-embedding-ada-002"
openai_api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter OPENAI_API_KEY: ")
os.environ["OPENAI_API_KEY"] = openai_api_key
openai_client = OpenAI()
print(f"      OpenAI client ready (embedding model: {embedding_model})", flush=True)

# Load vec2text corrector for inversion
print(f"[5/5] Loading vec2text corrector for OpenAI {embedding_model}...", flush=True)
print("      First run downloads ~1GB of model weights from HuggingFace.", flush=True)
print("      Subsequent runs load from the local HF cache (~/.cache/huggingface).", flush=True)
print("      This may take a few minutes depending on your hardware...", flush=True)
_t = time.time()
corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")
print(f"      Corrector loaded in {time.time() - _t:.1f}s — ready to invert.", flush=True)

In [ ]:
# 2. Set up ChromaDB and CyborgDB

import os
import shutil
import secrets
import uuid

import chromadb
from cyborgdb_core import Client, StorageConfig

# ChromaDB setup
persist_directory = "./tmp/chromadb_vuln_demo"
chroma_client = chromadb.PersistentClient(path=persist_directory)
try:
    chroma_client.delete_collection("sensitive_docs")
except Exception:
    pass
chroma_collection = chroma_client.create_collection("sensitive_docs")
print(f"✓ ChromaDB ready: {persist_directory}")

# CyborgDB setup — single-node disk-backed (RocksDB) encrypted index
CYBORGDB_DISK_PATH = "./tmp/cyborgdb_vuln_demo"
if os.path.exists(CYBORGDB_DISK_PATH):
    shutil.rmtree(CYBORGDB_DISK_PATH)
os.makedirs(CYBORGDB_DISK_PATH, exist_ok=True)

cyborgdb_client = Client(storage_config=StorageConfig.disk(CYBORGDB_DISK_PATH))

# dimension/metric auto-detect from the first upsert call.
cyborg_index = cyborgdb_client.create_index(
    index_name=f"comparison-demo-{uuid.uuid4()}",
    index_key=secrets.token_bytes(32),
)
print(f"✓ CyborgDB ready: {CYBORGDB_DISK_PATH} (disk-backed)")

In [ ]:
# 3. Define sensitive documents for demo

sensitive_documents = [
    # Financial/Identity Information
    "My social security number is 818-88-8118",
    "Password for admin account is SecurePass123!",
    
    # Medical Information
    "Patient diagnosed with Type 2 diabetes, prescribed metformin",
    "Blood test results show cholesterol level of 277 mg/dL",
    
    # Personal Information
    "John Doe lives at 1 World Trade Center, New York, NY 10007",
    "Phone number +1(212) 653-0688; email info@cyborg.co"
]

In [ ]:
# 4. Generate embeddings & store in BOTH databases

print("="*80)
print_colored("GENERATING EMBEDDINGS FROM SENSITIVE DOCUMENTS", bold=True)
print("="*80 + "\n")

def get_embeddings_openai(text_list):
    """Get embeddings from OpenAI API"""
    response = openai_client.embeddings.create(
        input=text_list,
        model=embedding_model,
        encoding_format="float",
    )
    return [e.embedding for e in response.data]

# Get embeddings (generate once, use for both databases)
embeddings = get_embeddings_openai(sensitive_documents)
print(f"Generated {len(embeddings)} embeddings of dimension {len(embeddings[0])}\n")

# Store in ChromaDB
chroma_collection.add(
    documents=sensitive_documents,
    embeddings=embeddings,
    ids=[f"sensitive_doc_{i}" for i in range(len(sensitive_documents))],
    metadatas=[{"type": "sensitive", "doc_num": i} for i in range(len(sensitive_documents))]
)
print("✓ Stored in ChromaDB")

# Store in CyborgDB
items = [
    {"id": f"sensitive_doc_{i}", "vector": embedding, "contents": doc}
    for i, (doc, embedding) in enumerate(zip(sensitive_documents, embeddings))
]
cyborg_index.upsert(items)
print("✓ Stored in CyborgDB (encrypted)")

In [ ]:
# 5. Extract embeddings from ChromaDB SQLite backend

import sqlite3
import struct

print("\n" + "="*80)
print_colored("EXTRACTING EMBEDDINGS FROM CHROMADB", bold=True)
print("="*80 + "\n")

# Connect to ChromaDB's SQLite database
db_path = os.path.join(persist_directory, "chroma.sqlite3")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get embedding data from database
cursor.execute("SELECT * FROM embeddings_queue ORDER BY seq_id")
rows = cursor.fetchall()

# Extract embeddings
chroma_extracted = []
for i, row in enumerate(rows):
    doc_id = row[4]
    embedding_blob = row[5]
    num_floats = len(embedding_blob) // 4
    embedding_values = struct.unpack(f'{num_floats}f', embedding_blob)
    chroma_extracted.append(list(embedding_values))
    print_colored(f"Extracted ChromaDB embedding {i+1}: {doc_id}, {len(embedding_values)} dimensions", "RED")

print(f"\n✓ Extracted {len(chroma_extracted)} embeddings from ChromaDB")

In [ ]:
# 6. Extract embeddings from CyborgDB on-disk store
#
# An attacker with filesystem access can read CyborgDB's RocksDB files
# directly. The strongest version of this demo opens the RocksDB store and
# pulls *individual* encrypted values straight from the keystore — one
# ciphertext blob per upserted vector, with no framing.
#
# CyborgDB stores each backing store (index/config/contents/vectors/metadata/id)
# as its own RocksDB directory under CYBORGDB_DISK_PATH. We find the one
# named `vectors` and iterate it.
#
# If direct opens fail (e.g., CyborgDB uses custom RocksDB options that block
# external opens), we fall back to an entropy-scan over raw bytes.

import re
import shutil
import tempfile
from collections import Counter

import numpy as np
import pathlib

print("\n" + "="*80)
print_colored("EXTRACTING EMBEDDINGS FROM CYBORGDB ON-DISK STORE", bold=True)
print("="*80 + "\n")

EMBED_DIM = 1536
EMBED_BYTES = EMBED_DIM * 4  # float32


def shannon_entropy(buf: bytes) -> float:
    if not buf:
        return 0.0
    counts = np.bincount(np.frombuffer(buf, dtype=np.uint8), minlength=256)
    probs = counts[counts > 0] / len(buf)
    return float(-np.sum(probs * np.log2(probs)))


def find_rocksdb_roots(base: pathlib.Path):
    """Return every directory under `base` that looks like a RocksDB store
    (i.e., contains a `CURRENT` file)."""
    return sorted({p.parent for p in base.rglob("CURRENT") if p.is_file()})


def extract_via_rocksdb(disk_path: str, needed: int):
    """Snapshot the on-disk store, locate each backing-store RocksDB root,
    and iterate the `vectors` keystore. Each value returned is an individual
    encrypted record."""
    from rocksdict import Rdict, Options, AccessType

    attacker_copy = pathlib.Path(tempfile.mkdtemp(prefix="rocksdb_exfil_"))
    shutil.copytree(disk_path, attacker_copy, dirs_exist_ok=True)
    print(f"  Snapshot copied to: {attacker_copy}")

    roots = find_rocksdb_roots(attacker_copy)
    if not roots:
        raise RuntimeError(f"No RocksDB roots (CURRENT files) under {attacker_copy}")
    print(f"  Found {len(roots)} RocksDB root(s):")
    for r in roots:
        size = sum(f.stat().st_size for f in r.iterdir() if f.is_file())
        print(f"    • {r.relative_to(attacker_copy)}  ({size:,} bytes)")

    # Prefer the keystore named 'vectors'; otherwise pick the largest root,
    # which most likely holds the bulk vector payloads.
    vectors_root = next((r for r in roots if r.name == "vectors"), None)
    if vectors_root is None:
        vectors_root = max(
            roots,
            key=lambda r: sum(f.stat().st_size for f in r.iterdir() if f.is_file()),
        )
        print_colored(
            f"  No subdir named 'vectors' — falling back to largest root: "
            f"{vectors_root.relative_to(attacker_copy)}",
            "YELLOW",
        )
    else:
        print(f"  Targeting: {vectors_root.relative_to(attacker_copy)}")

    # Each backing store is its own RocksDB DB → list_cf typically returns
    # just ["default"]. Open accordingly.
    try:
        cf_names = Rdict.list_cf(str(vectors_root))
    except Exception:
        cf_names = ["default"]
    print(f"  Column families in this root: {cf_names}")

    opts = Options(raw_mode=True)
    if cf_names == ["default"]:
        db = Rdict(
            path=str(vectors_root),
            options=opts,
            access_type=AccessType.read_only(),
        )
        cf = db
    else:
        cf_opts = {n: Options(raw_mode=True) for n in cf_names}
        db = Rdict(
            path=str(vectors_root),
            options=opts,
            access_type=AccessType.read_only(),
            column_families=cf_opts,
        )
        cf = db.get_column_family(
            "vectors" if "vectors" in cf_names else cf_names[0]
        )

    try:
        lengths = []
        out = []
        for key, value in cf.items():
            lengths.append(len(value))
            if len(out) >= needed:
                continue
            # Be generous on length — CEI may envelope ciphertext with a
            # nonce/tag. Accept anything within 64B of EMBED_BYTES.
            if abs(len(value) - EMBED_BYTES) > 64:
                continue
            blob = value[:EMBED_BYTES] if len(value) >= EMBED_BYTES else \
                   value.ljust(EMBED_BYTES, b"\x00")
            arr = np.frombuffer(blob, dtype=np.float32).copy()
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            out.append(arr.tolist())
            print_colored(
                f"  Extracted CyborgDB encrypted record {len(out)}: "
                f"key={key.hex()[:32]}…, value_len={len(value)} bytes "
                f"[ENCRYPTED — direct RocksDB read]",
                "GREEN",
            )
    finally:
        db.close()

    if lengths:
        print(f"  Total records in this keystore: {len(lengths)}")
        print(f"  Value-length histogram (top 5): {Counter(lengths).most_common(5)}")
    return out


def extract_via_entropy_scan(disk_path: str, needed: int):
    """Fallback: walk SST/WAL files raw, sample high-entropy windows."""
    DATA_FILE_RE = re.compile(r"(\.sst$|^\d+\.log$)", re.IGNORECASE)
    SCAN_STRIDE = 256
    ENTROPY_FLOOR = 7.0

    def file_score(p: pathlib.Path) -> float:
        try:
            return shannon_entropy(p.read_bytes()[:1 << 20])
        except OSError:
            return 0.0

    all_files = [p for p in pathlib.Path(disk_path).rglob("*")
                 if p.is_file() and p.stat().st_size >= EMBED_BYTES]
    candidate_files = sorted(
        (p for p in all_files if DATA_FILE_RE.search(p.name)),
        key=file_score, reverse=True,
    )
    print(f"  {len(all_files)} file(s) on disk, "
          f"{len(candidate_files)} match .sst/N.log pattern")

    out = []
    for path in candidate_files:
        if len(out) >= needed:
            break
        blob = path.read_bytes()
        offset = 0
        while offset <= len(blob) - EMBED_BYTES:
            chunk = blob[offset:offset + EMBED_BYTES]
            if shannon_entropy(chunk) < ENTROPY_FLOOR:
                offset += SCAN_STRIDE
                continue
            arr = np.frombuffer(chunk, dtype=np.float32).copy()
            if not np.all(np.isfinite(arr)):
                offset += SCAN_STRIDE
                continue
            out.append(arr.tolist())
            print_colored(
                f"  Extracted CyborgDB chunk {len(out)}: "
                f"{path.name} @ offset {offset} [ENCRYPTED — entropy scan]",
                "GREEN",
            )
            if len(out) >= needed:
                break
            offset += EMBED_BYTES

    if len(out) < needed:
        remaining = needed - len(out)
        sites = []
        for path in candidate_files:
            size = path.stat().st_size
            step = max(EMBED_BYTES, size // max(needed, 1))
            for offset in range(0, size - EMBED_BYTES + 1, step):
                sites.append((path, offset))
                if len(sites) >= needed * 4:
                    break
            if len(sites) >= needed * 4:
                break
        pick_stride = max(1, len(sites) // remaining) if sites else 1
        for path, offset in sites[::pick_stride][:remaining]:
            blob = path.read_bytes()[offset:offset + EMBED_BYTES]
            arr = np.frombuffer(blob, dtype=np.float32).copy()
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            out.append(arr.tolist())
            print_colored(
                f"  Extracted CyborgDB chunk {len(out)}: "
                f"{path.name} @ offset {offset} [ENCRYPTED — raw window]",
                "GREEN",
            )
    return out


needed = len(sensitive_documents)
cyborg_extracted = []

print_colored("Attempting direct RocksDB read of the 'vectors' keystore...", "YELLOW", bold=True)
try:
    cyborg_extracted = extract_via_rocksdb(CYBORGDB_DISK_PATH, needed)
    if cyborg_extracted:
        print_colored(
            f"\n✓ Direct RocksDB read recovered "
            f"{len(cyborg_extracted)} encrypted record(s) — clean ciphertext, no framing.",
            "GREEN", bold=True,
        )
except Exception as e:
    print_colored(
        f"  Direct RocksDB read failed: {type(e).__name__}: {e}",
        "RED",
    )
    print_colored("  Falling back to entropy-scan extraction.\n", "YELLOW")

if len(cyborg_extracted) < needed:
    print_colored(
        f"\nFalling back to entropy-scan for "
        f"{needed - len(cyborg_extracted)} remaining slot(s)...",
        "YELLOW",
    )
    cyborg_extracted.extend(
        extract_via_entropy_scan(CYBORGDB_DISK_PATH, needed - len(cyborg_extracted))
    )

if not cyborg_extracted:
    raise RuntimeError(
        "Could not extract any candidate chunks from disk. "
        "Has cell 4 been run, and does the disk path contain RocksDB files?"
    )

print(f"\n✓ Recovered {len(cyborg_extracted)} candidate chunk(s) from on-disk store")

In [ ]:
# 7. Define inversion function

import time
import torch

def run_inversion_attack(extracted_embeddings, db_name, color=""):
    """Run vec2text inversion attack on embeddings from a database"""
    
    print("\n" + "="*80)
    print_colored(f"{db_name.upper()}: RUNNING EMBEDDING INVERSION ATTACK", color, bold=True)
    print("="*80)
    
    # Convert to tensor
    embeddings_tensor = torch.tensor(extracted_embeddings, dtype=torch.float32)
    if torch.backends.mps.is_available():
        embeddings_tensor = embeddings_tensor.to('mps')
    elif torch.cuda.is_available():
        embeddings_tensor = embeddings_tensor.cuda()
    
    results = []
    
    for i, (original_doc, embedding) in enumerate(zip(sensitive_documents, embeddings_tensor)):
        print_colored(f"\n[{db_name}] Document #{i+1}:", bold=True)
        print(f"Original:      \"{original_doc}\"")
        
        start_time = time.time()
        reconstructed_list = vec2text.invert_embeddings(
            embeddings=embedding.unsqueeze(0),
            corrector=corrector,
            num_steps=4,
            sequence_beam_width=1,
        )
        reconstructed = reconstructed_list[0]
        inversion_time = time.time() - start_time
        
        print(f"Reconstructed: \"{reconstructed}\"")
        
        # Calculate similarity
        if len(reconstructed) > 0:
            orig_emb_cpu = embedding.cpu()
            new_emb = get_embeddings_openai([reconstructed])[0]
            new_emb_tensor = torch.tensor(new_emb)
            similarity = torch.nn.functional.cosine_similarity(orig_emb_cpu, new_emb_tensor, dim=0).item()
        else:
            similarity = 0
        
        exact_match = original_doc.lower().strip() == reconstructed.lower().strip()
        if exact_match:
            print_colored(f"Exact match!", "RED", bold=True)
        
        sim_color = "RED" if similarity > 0.99 else "YELLOW" if similarity > 0.95 else "GREEN"
        print_colored(f"Similarity: {similarity:.4f}", sim_color, bold=True)
        print(f"Time: {inversion_time:.2f}s")
        
        results.append({
            'original': original_doc,
            'reconstructed': reconstructed,
            'similarity': similarity,
            'time': inversion_time,
            'exact_match': exact_match,
        })
    
    return results

In [ ]:
# 8a. Run attack on ChromaDB

# Attack ChromaDB
chroma_results = run_inversion_attack(chroma_extracted, "ChromaDB", "RED")

In [ ]:
# 8b. Run attack on CyborgDB

# Attack CyborgDB  
cyborg_results = run_inversion_attack(cyborg_extracted, "CyborgDB", "GREEN")

In [ ]:
# 9. Side-by-Side Comparison Summary

print("\n" + "="*80)
print_colored("COMPARISON SUMMARY: ChromaDB vs CyborgDB", bold=True)
print("="*80 + "\n")

def calc_stats(results):
    total = len(results)
    exact = sum(1 for r in results if r['exact_match'])
    high_sim = sum(1 for r in results if r['similarity'] > 0.95)
    avg_sim = np.mean([r['similarity'] for r in results])
    avg_time = np.mean([r['time'] for r in results])
    return {
        'total': total,
        'exact': exact,
        'high_sim': high_sim,
        'avg_sim': avg_sim,
        'avg_time': avg_time
    }

chroma_stats = calc_stats(chroma_results)
cyborg_stats = calc_stats(cyborg_results)

print(f"{'Metric':<35} {'ChromaDB':>15} {'CyborgDB':>15}")
print("="*80)
print(f"{'Total documents':<35} {chroma_stats['total']:>15} {cyborg_stats['total']:>15}")
print(f"{'Exact reconstructions':<35} {chroma_stats['exact']:>15} {cyborg_stats['exact']:>15}")
print(f"{'High similarity (>95%)':<35} {chroma_stats['high_sim']:>15} {cyborg_stats['high_sim']:>15}")
print(f"{'Average similarity':<35} {chroma_stats['avg_sim']*100:>14.2f}% {cyborg_stats['avg_sim']*100:>14.2f}%")
print(f"{'Average inversion time':<35} {chroma_stats['avg_time']:>14.2f}s {cyborg_stats['avg_time']:>14.2f}s")

print("\n" + "="*80)
print_colored("CONCLUSION", bold=True)
print("="*80)

if chroma_stats['avg_sim'] > 0.95:
    print_colored("⚠ ChromaDB: VULNERABLE - High similarity indicates successful reconstruction", "RED", bold=True)
else:
    print_colored("✓ ChromaDB: Protected - Low similarity indicates failed reconstruction", "GREEN", bold=True)

if cyborg_stats['avg_sim'] > 0.95:
    print_colored("⚠ CyborgDB: VULNERABLE - High similarity indicates successful reconstruction", "RED", bold=True)
else:
    print_colored("✓ CyborgDB: PROTECTED - Low similarity indicates failed reconstruction", "GREEN", bold=True)

In [ ]:
# 10. Cleanup
chroma_client.delete_collection("sensitive_docs")
cyborg_index.delete_index()
shutil.rmtree(CYBORGDB_DISK_PATH, ignore_errors=True)